# Trade Discovery Pipeline Setup

Installs dependencies (once per session — skipped if already present), verifies
the DEAP GP engine, then runs the Global Evolutionary Loop (GEL) pipeline.

**Fresh-code guarantee:** every code cell purges project modules from the kernel,
deletes `__pycache__` / numba caches, and invalidates import caches before doing
anything — so re-running cells ALWAYS uses the latest cloned code. Installed
libraries (ta-lib, vectorbt, deap, numpy, …) are never removed or rebuilt
unless actually missing.

In [ ]:
# ── Dependencies: install ONLY what's missing (never reinstall/rebuild) ──────
import importlib, importlib.util, os

%cd /kaggle/working

def _have(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except Exception:
        return False

# 1) TA-Lib (C library build is SLOW — skip entirely if the wrapper imports)
if _have("talib"):
    print("✅ ta-lib already installed — skipping C build & pip install")
else:
    print("Installing TA-Lib C library + wrapper ...")
    if not os.path.exists("ta-lib-0.4.0-src.tar.gz"):
        !wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
    !tar -xzf ta-lib-0.4.0-src.tar.gz
    %cd ta-lib/
    !./configure --prefix=/usr
    !make
    !make install
    %cd /kaggle/working
    !ldconfig
    !rm -rf ta-lib ta-lib-0.4.0-src.tar.gz
    !pip install -q ta-lib

# 2) Core Python deps — install only the missing/mismatched ones
need = []
if not _have("vectorbt"):            need.append("vectorbt")
if not _have("deap"):               need.append("deap==1.4.4")
if not _have("pyarrow"):             need.append("pyarrow")
if not _have("numba"):               need.append("numba")
_sk = importlib.util.find_spec("scikit-learn") or _have("sklearn")
if _have("sklearn"):
    import sklearn
    maj, mino = (int(x) for x in sklearn.__version__.split(".")[:2])
    if (maj, mino) >= (1, 9):
        need.append('"scikit-learn<1.9"')
elif _sk:
    need.append("scikit-learn<1.9")

if need:
    print("Installing missing packages:", need)
    !pip install -q {" ".join(need)}
else:
    print("✅ all core dependencies present — nothing installed")

In [ ]:
# ── Fresh clone + project-cache purge utility ────────────────────────────────
import os, sys, shutil, importlib

REPO_URL = "https://github.com/ayan1-git/gplearn-2.git"
BRANCH   = "gplearn"
REPO_DIR = "gplearn-2"


def purge_project_caches(project_root=None):
    """Evict EVERYTHING project-level that can be stale, keeping installed
    libraries untouched:
      1. project modules cached in sys.modules (the #1 cause of 'old code runs')
      2. sys.path entries pointing inside the repo
      3. import-system caches
      4. __pycache__ dirs and numba (.nbc/.nbi) JIT caches under the repo
    """
    root = os.path.realpath(project_root or os.getcwd())
    evicted = []
    for name, mod in list(sys.modules.items()):
        f = getattr(mod, "__file__", None)
        if f:
            try:
                if os.path.realpath(f).startswith(root + os.sep):
                    del sys.modules[name]
                    evicted.append(name)
            except OSError:
                pass
    sys.path[:] = [p for p in sys.path
                   if not (p and os.path.realpath(p).startswith(root + os.sep))]
    importlib.invalidate_caches()
    n_numba = 0
    for dirpath, dirnames, filenames in os.walk(root):
        for d in list(dirnames):
            if d == "__pycache__":
                shutil.rmtree(os.path.join(dirpath, d), ignore_errors=True)
                dirnames.remove(d)
        for fn in filenames:
            if fn.endswith((".nbc", ".nbi")):
                try:
                    os.remove(os.path.join(dirpath, fn))
                    n_numba += 1
                except OSError:
                    pass
    print(f"♻️  purged {len(evicted)} cached module(s) "
          f"+ __pycache__ + {n_numba} numba cache file(s)")
    return evicted


%cd /kaggle/working

# Purge caches of ANY previous clone BEFORE deleting it (old tree still
# exists, so the walk can clean it; modules from the old clone get evicted).
if os.path.exists(REPO_DIR):
    purge_project_caches(os.path.join("/kaggle/working", REPO_DIR))
    print(f"Removing existing clone: {REPO_DIR} ...")
    shutil.rmtree(REPO_DIR)

print(f"Cloning {REPO_URL} (branch: {BRANCH}) ...")
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

# Automated discovery of the project root
src_path = None
for root, dirs, files in os.walk(f"/kaggle/working/{REPO_DIR}"):
    if 'src' in dirs:
        src_path = os.path.join(root, 'src')
        break

if src_path:
    project_root = os.path.dirname(src_path)
    print(f"Project root found: {project_root}")
    %cd {project_root}
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
else:
    print("❌ 'src' directory not found!")
    !find /kaggle/working/{REPO_DIR} -maxdepth 3

print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Verify data existence and config — ALWAYS on freshly imported code
try:
    purge_project_caches()          # evict src.* from kernel, clear __pycache__
except NameError:
    pass                            # cell 2 not run yet
import os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import src.config as cfg           # guaranteed-fresh import

print(f"Configured DATAPATH: {cfg.DATAPATH}")
if os.path.exists(cfg.DATAPATH):
    print(f"✅ Data file found: {cfg.DATAPATH}")
    !ls -lh "{cfg.DATAPATH}"
else:
    print(f"❌ Data file NOT found at {cfg.DATAPATH}")
    print("Full path check:", os.path.abspath(cfg.DATAPATH))
    print("\nDirectory structure (depth 2):")
    !ls -R data/

In [ ]:
# Pre-flight: verify the DEAP GP engine + all pipeline fixes BEFORE burning
# hours on the full GEL loop. Raises (stops the run) if anything fails.
try:
    purge_project_caches()
except NameError:
    pass

import subprocess, sys
_r = subprocess.run([sys.executable, "-B", "scripts/verify_fixes.py"])
if _r.returncode != 0:
    raise RuntimeError("verify_fixes.py FAILED \u2014 resolve the failing "
                       "checks before running the GEL pipeline.")

In [ ]:
# Run the GEL Pipeline — subprocess is always fresh; -B prevents any new
# __pycache__ from being written, so nothing can go stale across runs.
try:
    purge_project_caches()
except NameError:
    pass

!python -B scripts/main_pipeline.py